# 04 — Normalization (Part D, task 5)

In [ ]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
%load_ext autoreload
%autoreload 2

from src.pretrained import load_pretrained
pretrained = load_pretrained()     # provided helper: {"gpt2": ..., "bert-base-uncased": ..., "google/flan-t5-small": ...}
assert pretrained, "No pretrained tokenizer could be loaded: check your internet connection."

## Compare the pretrained tokenizers

In [ ]:
sentences = ["HELLO WORLD", "Hello World", "hello world", "Café", "Cafe", "CAFÉ", "I'm learning NLP!", "I'm learning NLP!!!"]
pd.DataFrame([{"sentence": s, **{name: t.tokenize(s) for name, t in pretrained.items()}} for s in sentences])

**Observe:** which pairs of inputs give *identical* tokens for which tokenizer?

### 🧩 Exercise 4.1 — Ask a tokenizer for its normalizer

Get the normalizer of each fast tokenizer and apply it to a string to see what it changes.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `tok.backend_tokenizer.normalizer` (may be `None`).
* A normalizer object has `normalize_str(text)`.

In [ ]:
for name, t in pretrained.items():
    norm = t.backend_tokenizer.____
    print(f"{name:22} normalizer = {type(norm).__name__ if norm else None}")
    if norm is not None:
        for s in ["HELLO Café!", "e\u0301", "ﬁnance"]:
            print(f"{'':25} {s!r:14} -> {norm.____(s)!r}")

In [ ]:
print("✅ Exercise 4.1 ran (look at which models have a normalizer, and what each does)")

## Build normalizers yourself (offline)

### 🧩 Exercise 4.2 — Compose normalizers

Build a normalizer that **decomposes** characters, **removes accents**, then **lowercases**; and the BERT-style one with the same effect.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* Classes in `tokenizers.normalizers`: `NFD`, `StripAccents`, `Lowercase`, `Sequence`, `BertNormalizer`.
* `Sequence([a, b, c])` applies the normalizers in order. Why must `NFD()` come *before* `StripAccents()`? (see the walkthrough, §5.1)
* `BertNormalizer(lowercase=..., strip_accents=...)` does the same in a single object.
* Docs: https://huggingface.co/docs/tokenizers/api/normalizers

In [ ]:
from tokenizers.normalizers import NFC, NFD, NFKC, Lowercase, StripAccents, BertNormalizer, Sequence

my_norm   = Sequence([____, ____, ____])
bert_norm = BertNormalizer(____, ____)

for s in ["Ñandú CAFÉ", "Zażółć"]:
    print(repr(s), "->", my_norm.normalize_str(s), "|", bert_norm.normalize_str(s))

In [ ]:
assert my_norm.normalize_str("CAFÉ") == "cafe"
assert bert_norm.normalize_str("CAFÉ") == "cafe"
print("✅ Exercise 4.2 OK")

In [ ]:
samples = ["HELLO Café", "e\u0301 (decomposed é)", "\ufb01nance", "السَّلامُ", "  spaces  "]
pipelines = {"NFC": NFC(), "NFD": NFD(), "NFKC": NFKC(), "Lowercase": Lowercase(),
             "NFD+StripAccents": Sequence([NFD(), StripAccents()]), "BertNormalizer": bert_norm}
pd.DataFrame([{"input": s, **{n: p.normalize_str(s) for n, p in pipelines.items()}} for s in samples]).T

**Questions**
1. What is a tokenizer normalizer?
2. Lowercasing vs. Unicode normalization?
3. Why does BERT-uncased behave differently from GPT-2?
4. Why must normalization depend on the tokenizer and its training? (look at what happened to the Arabic example in the table)

**Your answer:**

_(write here)_